# OE5: reproducción autónoma en Google Colab

**Objetivo oficial (literal).** Determinar el nivel de precisión de la adaptación multimodal del artefacto desarrollado, calculado automáticamente como F1_adapt frente a la referencia gold, contrastándolo con el valor de referencia establecido de 0,85.

**Qué comprueba este cuaderno.** Recalcula el F1 por caso desde el vector de selección de cada ciclo y la referencia gold sellada, el F1_adapt por perfil, los intervalos, las pruebas unilaterales, la Enmienda 1 y la sensibilidad a la regla de inclusión, y los compara con el resultado local. Reproducir el cálculo no es volver a generar los 1 000 ciclos.

**Datos de entrada.** Ciclos de la campaña (10 réplicas por perfil con el vector de selección), plan, perfiles, referencia gold sellada y los dos scripts de análisis, con los resultados locales para comparar.

**Qué no hace.** No regenera los 1 000 ciclos ni verifica la independencia del F1 respecto del equipo. No repite las campañas ni regenera datos, no modifica archivos originales, scripts ni umbrales, y no convierte ninguna campaña en controlada. Pulse «Ejecutar todo» en un entorno nuevo de Colab: los datos se descargan solos desde GitHub (commit fijado) y se verifican por SHA-256; no hay que subir ni buscar archivos.

**Contenido.** 1 descripción · 2 entorno y código (clonado de la etiqueta inmutable) · 3 descarga automática y verificación SHA-256 · 4 validación de integridad y estructura · 5 preprocesamiento y cálculo de indicadores (scripts sellados) · 6 análisis estadístico · 7 comparación con el resultado local y con las cifras de la tesis · 8 tablas · 9 gráficos · 10 pruebas automáticas · 11 resumen. Las diferencias se clasifican como exacta, dentro de tolerancia, explicada por versión, no explicada o no reproducible.

## 1. Entorno y código de análisis
El código vive en el repositorio público (módulos legibles en `reproducibility/etapa26_colab_autonomo/src/repro/`); los scripts de análisis sellados vienen dentro de los paquetes de datos, con su hash verificado.

In [ ]:
import os, sys, platform, subprocess, json
IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/Renato5Lara/multiagent-llm-education.git'
REPO_REF = 'repro-etapa26-v3'            # etiqueta inmutable del codigo; los datos estan fijados por commit en manifests/data_manifest.json
if os.environ.get('REPRO_LOCAL_REPO'):   # solo para pruebas locales antes de publicar la etiqueta
    REPO_DIR = os.environ['REPRO_LOCAL_REPO']
else:
    REPO_DIR = 'repo_repro'
    if not os.path.isdir(REPO_DIR):
        subprocess.run(['git', 'clone', '--quiet', '--depth', '1', '--branch', REPO_REF, '--filter=blob:none', '--sparse', REPO_URL, REPO_DIR], check=True)
        subprocess.run(['git', '-C', REPO_DIR, 'sparse-checkout', 'set', 'reproducibility/etapa26_colab_autonomo'], check=True)   # solo el codigo de esta etapa
sys.path.insert(0, os.path.join(REPO_DIR, 'reproducibility', 'etapa26_colab_autonomo', 'src'))
import repro
from repro import sh
print('Codigo de analisis:', REPO_URL, '@', REPO_REF)


In [ ]:
import numpy, scipy, pandas, matplotlib
ENTORNO = dict(python=platform.python_version(), plataforma=platform.platform(), numpy=numpy.__version__, scipy=scipy.__version__, pandas=pandas.__version__, matplotlib=matplotlib.__version__, cpu=os.cpu_count())
print(ENTORNO)


## 2. Descarga automática y verificación de los paquetes (SHA-256)

In [ ]:
# Descarga automatica desde GitHub (commit fijado) y verificacion SHA-256 contra el manifiesto. No hay que subir nada.
# Si falta un archivo, no se puede descargar o su hash no coincide, se detiene con el archivo, la URL y el hash esperado.
PAQUETES = ['oe5_colab_paquete']
repro.fetch_packages(PAQUETES, dest='oe5')


## 3. Verificación interna, cálculo y análisis (celdas del notebook de comprobación publicado, sin cambios salvo la ejecución de scripts)

In [ ]:
import hashlib
base='oe5/campana_01'; bad=[]
for l in open(base+'/SHA256SUMS.txt'):
    h,n=l.split(None,1); n=n.strip().lstrip('*')
    if hashlib.sha256(open(os.path.join(base,n),'rb').read()).hexdigest()!=h: bad.append(n)
print('archivos con hash distinto:', bad); assert not bad

In [ ]:
sh(f"python oe5/analisis_oe5.py oe5/campana_01 oe5/salida_colab > oe5/log_analisis.txt 2>&1")
sh(f"python oe5/analisis_oe5_enmienda1.py oe5/campana_01 oe5/salida_colab_enmienda1 > oe5/log_enmienda1.txt 2>&1")
for _l in ['oe5/log_analisis.txt', 'oe5/log_enmienda1.txt']: print(_l, "->", open(_l, encoding="utf-8", errors="replace").read()[-300:])

In [ ]:
import json
a=json.load(open('oe5/salida_colab/resultados_oe5.json')); b=json.load(open('oe5/esperado_resultados_oe5_local.json'))
for k in ['media','de','dz','shapiro_p','t_p_mayor','t_p_menor','wilcoxon_p_mayor']:
    print(k, a['principal'][k], b['principal'][k], 'dif=%.2e'%abs(a['principal'][k]-b['principal'][k]))
print('IC95 t', a['principal']['ic95_t'], b['principal']['ic95_t'])
print('bootstrap IC (semilla fija)', a['principal']['ic95_bootstrap'], b['principal']['ic95_bootstrap'])
print('sensibilidad', a['sensibilidad_regla_inclusion'])
c=json.load(open('oe5/salida_colab_enmienda1/resultados_enmienda1.json')); d=json.load(open('oe5/esperado_enmienda1_local.json'))
for k in ['p_bootstrap_t_mayor','p_t_mayor']: print(k, c['prueba_de_la_media'][k], d['prueba_de_la_media'][k])
print('ICC', c['dependencia_replicas']['icc'], d['dependencia_replicas']['icc'])
print('HE5 respaldada:', c['decision']['HE5_mu_mayor_0.85_respaldada'], '| RNF03 cumplido:', c['decision']['RNF03_cumplido'])
print('valor principal conservado:', c['decision']['valor_principal_conservado'])

## 4. Comparación con las cifras de la tesis
Cada cifra se lee del resultado **reproducido** y se compara con la que figura en la tesis (versión CIERRE_FINAL) y en el JSON oficial sellado (`manifests/valores_oficiales.json`).

In [ ]:
from repro import official, validation, tables, figures
from repro.data_loading import sha256_file
import pandas as pd
RUTAS = {'resultados_oe5.json': 'oe5/salida_colab/resultados_oe5.json', 'f1_por_perfil.csv': 'oe5/salida_colab/f1_por_perfil.csv'}
cifras = official.evaluar('OE5', RUTAS)
print(cifras[['id', 'descripcion', 'esperado', 'obtenido', 'dif', 'estado']].to_string(index=False))


## 5. Clasificación de las diferencias frente al resultado local
Exacta · dentro de tolerancia (1e-9) · explicada por versión · no explicada · no reproducible.

In [ ]:
import scipy
REG = validation.Registro(scipy_version=scipy.__version__)
COMPARACIONES = [('OE5 resultados_oe5.json', 'oe5/salida_colab/resultados_oe5.json', 'oe5/esperado_resultados_oe5_local.json'), ('OE5 enmienda 1', 'oe5/salida_colab_enmienda1/resultados_enmienda1.json', 'oe5/esperado_enmienda1_local.json')]
for _e, _a, _b in COMPARACIONES:
    REG.comparar(_e, _a, _b)


## 6. Tablas

In [ ]:
OUT = 'outputs'
TABLAS = {'f1_por_perfil': 'oe5/salida_colab/f1_por_perfil.csv'}
tablas_generadas = tables.exportar_tablas('OE5', TABLAS, os.path.join(OUT, 'tables'))
print('Tablas generadas:')
for _t in tablas_generadas:
    print('  ', _t)


## 7. Gráficos
Se generan desde la salida del análisis ejecutado arriba (no desde valores copiados). Se exportan PNG (300 ppi), SVG y el CSV de la serie dibujada (`figure_data/`).

In [ ]:
METAS = official.cargar_valores()['metas_de_diseno']
fig = figures.fig_oe5(RUTAS['f1_por_perfil.csv'], os.path.join(OUT, 'figures', 'OE5'), metas=METAS)
_of = official.cargar_valores()
CSV_IDENTICO = sha256_file(RUTAS['f1_por_perfil.csv']) == _of['entradas_de_figuras_oficiales']['f1_por_perfil_csv_sha256_oficial']
print('f1_por_perfil.csv reproducido: SHA-256 identico al del resultado oficial que alimento la Figura 5:', CSV_IDENTICO)
_t = _of['figuras_tesis']['fig_oe5']
IDENTICA = sha256_file(fig['png']) == _t['sha256_png_tesis']
print('Figura 5: PNG identico bit a bit al de la tesis (pagina %s del indice): %s' % (_t['pagina_indice'], IDENTICA))
if not IDENTICA:
    print('  Esperable fuera del entorno de la tesis (matplotlib 3.10.3 y fuente Arial); la comprobacion valida es la del CSV de entrada y la serie dibujada.')
try:
    from IPython.display import Image, display
    display(Image(fig['png']))
except Exception:
    pass


## 8. Pruebas automáticas de consistencia

In [ ]:
fallos = []
if cifras.estado.isin(['DIFIERE', 'NO_EVALUABLE']).any():
    fallos.append('cifras oficiales: ' + str(cifras[cifras.estado.isin(['DIFIERE', 'NO_EVALUABLE'])][['id', 'estado']].values.tolist()))
peor, cuenta = REG.veredicto()
if peor in ('NO_EXPLICADA', 'NO_REPRODUCIBLE'):
    fallos.append('comparacion con el resultado local: ' + peor)
if not (os.path.getsize(fig['png']) > 0 and os.path.getsize(fig['serie_csv']) > 0):
    fallos.append('figura vacia')
if not (CSV_IDENTICO or (cifras.loc[cifras.id == 'OE5-01', 'estado'] != 'DIFIERE').all()):
    fallos.append('f1_por_perfil.csv difiere del oficial')
assert not fallos, fallos
print('PRUEBAS AUTOMATICAS DE CONSISTENCIA: todas superadas | cifras oficiales comprobadas:', len(cifras), '| comparaciones:', cuenta)


## 9. Resumen final de la reproducción

In [ ]:
import shutil
resumen = tables.guardar_resumen('OE5', os.path.join(OUT, 'reports'), REG.tabla(), cifras, ENTORNO, extra=dict(veredicto=peor, conteo=cuenta, paquetes=PAQUETES))
print('RESUMEN DE REPRODUCCION OE5')
print(REG.tabla()[['comparacion', 'categoria', 'dif_abs_max']].to_string(index=False))
print('Veredicto (peor categoria):', peor, cuenta)
print('Cifras de la tesis comprobadas: %d (%d coinciden exactamente, %d dentro del redondeo)' % (len(cifras), (cifras.estado == 'COINCIDE').sum(), (cifras.estado == 'COINCIDE_REDONDEO').sum()))
print('LIMITACIONES: F1_adapt 0,8044 (IC95 % 0,7785 a 0,8303) no alcanza el umbral de 0,85 (RNF03 no cumplido); 100 perfiles sinteticos x 10 repeticiones; la dependencia entre repeticiones se analiza en la enmienda 1.')
print('Alcance: se reproduce el analisis, no la adquisicion de datos; no se repite ninguna campana.')
_z = shutil.make_archive('resultados_OE5', 'zip', OUT)
print('Salidas empaquetadas en', os.path.basename(_z))
if IN_COLAB:
    from google.colab import files
    files.download(_z)
